# Serving, Policy y auditoría: del modelo a una evidencia verificable

Ejecuta primero el Notebook 04. Allí se decide qué **versión** es `champion`; aquí comprobamos qué ocurre cuando un proceso real la carga y atiende solicitudes. Antes del código, separa estos cuatro sustantivos:

| Concepto | Pregunta que responde | No es |
|---|---|---|
| **Registry** | ¿Qué versión registrada apunta hoy a `champion`? | Un servidor ejecutándose. |
| **Model API** | ¿Qué copia del modelo cargó este proceso al arrancar? | El Registry ni una regla de negocio. |
| **Policy** | Dada una probabilidad, ¿qué recomendación contractual corresponde? | La probabilidad ni la decisión final. |
| **Audit** | ¿Qué evidencia deja cada evaluación? | Una métrica de MLflow ni una decisión final. |

Ruta que observaremos: `champion` en Registry -> Model API -> probabilidad -> Policy -> `model_evaluations`. El Registry y el runtime se mantienen separados deliberadamente: la API carga una sola vez al iniciar.

> **Precondición operativa.** En el aula Compose ya ejecuta la Model API permanente. Este notebook la consume mediante `INVOICEOPS_MODEL_API_URL` (por defecto `http://model-api:8001` dentro de Docker); no inicia ni detiene procesos. Las celdas de auditoría escriben SQLite; `/health` y `/predict` son lecturas de la API.

In [ ]:
# Side-effect contract: resource=local state read only; condition=Notebook 04 state exists; idempotency=no write in this preflight; recovery=complete 04 or inspect the declared local state.
import contextlib
import importlib
import io
import json
import os
import sys
from pathlib import Path

import httpx
import mlflow
import pandas as pd
from IPython.display import Markdown, display
from mlflow.tracking import MlflowClient

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
# El override conserva solo estado técnico; SQLite operacional viene de INVOICEOPS_DB_PATH.
DEMO_ROOT = Path(
    os.environ.get("INVOICEOPS_NOTEBOOK_DEMO_ROOT", PROJECT_ROOT / "var" / "local-demo" / "notebook-state")
).resolve()
STATE_PATH = DEMO_ROOT / "state.json"
if not STATE_PATH.exists():
    raise RuntimeError("Ejecuta primero el Notebook 04 con el mismo INVOICEOPS_NOTEBOOK_DEMO_ROOT.")
state = json.loads(STATE_PATH.read_text())
state.setdefault("completed_actions", {})
state.setdefault("evaluations", {})
TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI")
MLFLOW_UI_URL = os.environ.get("INVOICEOPS_MLFLOW_UI_URL")
if not TRACKING_URI:
    raise RuntimeError(
        "Preflight MLflow: falta MLFLOW_TRACKING_URI en este kernel. Inicia el servidor compartido, exporta la URI y reinicia el kernel antes de continuar."
    )
if not MLFLOW_UI_URL:
    raise RuntimeError(
        "Preflight MLflow: falta INVOICEOPS_MLFLOW_UI_URL en este kernel. Configura la URL del navegador y reinicia el kernel antes de continuar."
    )
resolve_db_path = importlib.import_module("invoiceops.legacy.db").resolve_db_path

DEMO_DB = resolve_db_path(None)
AUDIT_ACTION_SCOPE = f"db:{DEMO_DB.resolve()}"
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()
try:
    client.search_experiments(max_results=1)
except Exception as error:
    raise RuntimeError(
        f"Preflight MLflow: no se puede conectar a {TRACKING_URI}. Verifica que el único servidor MLflow compartido esté activo y que la URI sea accesible. Detalle: {error}"
    ) from error
display(
    Markdown(
        f"**Backend MLflow compartido preparado.** Tracking/Registry: `{TRACKING_URI}`. Estado técnico: `{STATE_PATH.name}`. Auditoría SQLite operacional: `{DEMO_DB}`."
    )
)

## 1. Resolver el champion

**Qué estás mirando:** el alias móvil `champion` del Registry y la identidad de la versión a la que apunta ahora.

**Por qué importa:** un alias permite hablar de la versión aprobada sin escribir un número fijo. Sin embargo, saber el alias no prueba todavía qué modelo usa un proceso en memoria.

**Qué ejecuta esta celda:** consulta MLflow para resolver `models:/invoice-review@champion` y recupera su run.

**Qué deberías comprobar:** nombre, versión, run y tipo del modelo aparecen en una tabla.

**Qué significa si falla:** 04 no terminó, el alias no existe o este notebook no usa el mismo `DEMO_ROOT`. No inventes una versión: vuelve a verificar el Registry.

In [ ]:
from invoiceops.ml.registry import MODEL_NAME

champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
champion_run = client.get_run(champion.run_id)
champion_table = pd.DataFrame(
    [
        {
            "Nombre": MODEL_NAME,
            "Versión champion": f"v{champion.version}",
            "Run": champion.run_id,
            "Tipo": champion_run.data.params.get("model_type", "desconocido"),
        }
    ]
)
display(champion_table)

## Ver esta ejecución en MLflow

MLflow UI es un **visor del mismo backend compartido** que usa esta demo; no crea una copia de runs ni del Registry. Inicia ese único servidor siguiendo el README antes de abrir Jupyter: no inicies otro servidor.

En la misma UI revisa **Experiments -> invoice-risk -> runs** para métricas, parámetros y artifacts, y **Models -> invoice-review** para versiones y aliases. Este notebook escribe `model_evaluations`, `source`, `reason` y `correlation_id` en la SQLite operacional compartida resuelta por `INVOICEOPS_DB_PATH`; se observan en las tablas del notebook o el portal, **no** en MLflow.

In [ ]:
print(f"Abre la UI de MLflow: {MLFLOW_UI_URL}")
print("Experiments -> invoice-risk -> runs; Models -> invoice-review -> versiones y aliases.")

## 2. Ejercitar la Model API de Compose y confirmar health

**Qué estás mirando:** el servicio Model API permanente de Compose y la identidad del modelo que cargó al iniciar.

**Por qué importa:** Registry state no es Runtime state. Cambiar `champion` después no hace hot reload; solo un reinicio puede cargar la nueva referencia.

**Qué ejecuta esta celda:** lee `INVOICEOPS_MODEL_API_URL`, consulta `/health` del servicio Compose y muestra la respuesta. No crea un segundo Uvicorn ni modifica el servicio.

**Qué deberías comprobar:** `status=ok`, nombre, versión y run deben coincidir con la tabla anterior.

**Qué significa si falla:** el servicio Compose no está accesible o no quedó sano. No continúes a `/predict`; revisa `INVOICEOPS_MODEL_API_URL`, el estado de Compose y el Registry.

In [ ]:
# Side-effect contract: resource=Compose Model API read only; condition=service healthy; idempotency=no write; recovery=inspect the configured URL and Compose status.
MODEL_API_BASE_URL = os.environ.get("INVOICEOPS_MODEL_API_URL", "http://model-api:8001").rstrip("/")
if not MODEL_API_BASE_URL:
    raise RuntimeError("Preflight Model API: INVOICEOPS_MODEL_API_URL no puede estar vacía.")
BASE_URL = MODEL_API_BASE_URL
try:
    health_response = httpx.get(f"{BASE_URL}/health", timeout=5)
    health_response.raise_for_status()
except httpx.HTTPError as error:
    raise RuntimeError(
        f"Preflight Model API: no se puede consultar {BASE_URL}/health. Verifica INVOICEOPS_MODEL_API_URL y Compose. Detalle: {error}"
    ) from error
health = health_response.json()
display(
    pd.DataFrame(
        [
            {
                "Estado": health["status"],
                "Nombre": health["model_name"],
                "Versión cargada": f"v{health['model_version']}",
                "Run cargado": health["run_id"],
            }
        ]
    )
)

## 3. Features, `/predict` y tres mecanismos distintos

**Qué estás mirando:** para `INV-10029` e `INV-10030`, la recomendación de Rule v1, la probabilidad devuelta por el modelo y la recomendación final de Policy.

**Por qué importa:** Rule v1 usa reglas explícitas; el modelo estima una probabilidad con ocho features; Policy transforma esa probabilidad mediante su umbral contractual. Son mecanismos diferentes aunque puedan coincidir. Los resultados son datos reales de esta ejecución: ninguna factura tiene una decisión ML prefijada.

**Qué ejecuta esta celda:** inicializa la SQLite operacional compartida resuelta por `INVOICEOPS_DB_PATH`, transforma cada factura con `invoice_to_features`, llama `/predict` y aplica `recommend_from_probability`. El estado técnico auxiliar permanece separado.

**Qué deberías comprobar:** compara cada fila, incluyendo versión/run del modelo, Rule v1, probabilidad y Policy. La igualdad o diferencia entre recomendaciones se interpreta a partir de la salida real.

**Qué significa si falla:** un contrato de features o la API está fallando; no sustituyas la respuesta por una probabilidad inventada.

In [ ]:
# Side-effect contract: resource=isolated demo SQLite seed; condition=missing invoices; idempotency=init/seed only fills missing demo data; recovery=reset the declared local-demo root, never shared state.
from invoiceops.domain.policy import fallback_recommendation, recommend_from_probability
from invoiceops.domain.rules import decide_invoice
from invoiceops.legacy.db import (
    get_invoice,
    init_db,
    insert_model_evaluation,
    list_model_evaluations,
)
from invoiceops.legacy.seed import seed_invoices
from invoiceops.ml.features import invoice_to_features

with contextlib.redirect_stdout(io.StringIO()):
    init_db(DEMO_DB)
if get_invoice(DEMO_DB, "INV-10029") is None:
    seed_invoices(DEMO_DB)


def predict_and_recommend(invoice_id):
    invoice = get_invoice(DEMO_DB, invoice_id)
    response = httpx.post(f"{BASE_URL}/predict", json=invoice_to_features(invoice), timeout=5)
    response.raise_for_status()
    prediction = response.json()
    return invoice, prediction, recommend_from_probability(prediction["manual_review_probability"])


comparison_rows = []
for invoice_id in ("INV-10029", "INV-10030"):
    invoice, prediction, recommendation = predict_and_recommend(invoice_id)
    comparison_rows.append(
        {
            "Factura": invoice_id,
            "Rule v1": decide_invoice(invoice).value,
            "Probabilidad": prediction["manual_review_probability"],
            "Policy": recommendation.decision.value,
            "Versión Policy": recommendation.policy_version,
            "Versión modelo": f"v{prediction['model_version']}",
            "Run": prediction["run_id"],
        }
    )
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table.style.format({"Probabilidad": "{:.3f}"}))
print("Comparación completada con respuestas reales de /predict; no hay decisiones ML prefijadas.")

## 4. Metadata del runtime Compose y evidencia persistida

**Qué estás mirando:** la metadata que el runtime Compose declara en `/health` y la misma metadata que devuelve `/predict` antes de persistir una evaluación.

**Por qué importa:** una Promotion cambia Registry, no la memoria del servidor. Este notebook no reinicia servicios: `/health` es la evidencia de la versión que el runtime Compose ya tiene cargada.

**Qué ejecuta esta celda:** **⚠ MODIFICA ESTADO** solamente en SQLite operacional y `STATE_PATH`; lee `/health`, llama `/predict` y persiste una evaluación del runtime actual. Sus acciones son idempotentes: una segunda ejecución no duplica auditorías. Recuperación: relee `STATE_PATH`, auditoría y `/health`; corrige el preflight sin repetir a ciegas.

**Qué deberías comprobar:** `/health` y `/predict` muestran la misma versión y run; la tabla de auditoría añade `source`, `reason` y `correlation_id`.

**Qué significa si falla:** si `/health` y `/predict` no coinciden, hay una discrepancia de runtime y no se debe persistir como si la evaluación fuera trazable.

In [ ]:
# Side-effect contract: resource=Compose Model API read plus SQLite audit and local state writes; condition=healthy runtime; idempotency=durable action identity; recovery=inspect state, audit, and health before retrying.
# ⚠ MODIFICA ESTADO: auditoría operacional protegida por identidad durable.
from notebooks.demo_helpers import run_mutable_action_once


def save_state():
    STATE_PATH.write_text(json.dumps(state, indent=2, sort_keys=True) + "\n")


def persist_runtime_evaluation():
    runtime = httpx.get(f"{BASE_URL}/health", timeout=5).json()
    invoice, prediction, recommendation = predict_and_recommend("INV-10030")
    if (prediction["model_version"], prediction["run_id"]) != (
        runtime["model_version"],
        runtime["run_id"],
    ):
        raise RuntimeError("La metadata de /health y /predict no coincide; no se persistió auditoría.")
    correlation_id = f"notebook-05:{invoice.invoice_id}:compose-runtime:v{prediction['model_version']}"

    def persist_once():
        insert_model_evaluation(
            DEMO_DB,
            invoice.invoice_id,
            correlation_id=correlation_id,
            recommendation=recommendation,
            model_name=prediction["model_name"],
            model_version=prediction["model_version"],
            run_id=prediction["run_id"],
            manual_review_probability=prediction["manual_review_probability"],
        )
        return {
            "Runtime": BASE_URL,
            "Versión": f"v{prediction['model_version']}",
            "Run": prediction["run_id"],
            "Probabilidad": prediction["manual_review_probability"],
            "Policy": recommendation.decision.value,
        }

    state["evaluations"]["compose-runtime"] = run_mutable_action_once(
        f"persist-INV-10030-compose-runtime:{AUDIT_ACTION_SCOPE}",
        state["completed_actions"],
        persist_once,
    )
    save_state()


persist_runtime_evaluation()
display(pd.DataFrame([state["evaluations"]["compose-runtime"]]).style.format({"Probabilidad": "{:.3f}"}))
audit = pd.DataFrame([dict(row) for row in list_model_evaluations(DEMO_DB, "INV-10030")])
display(
    audit[
        [
            "invoice_id",
            "model_name",
            "model_version",
            "run_id",
            "manual_review_probability",
            "recommendation",
            "source",
            "reason",
            "correlation_id",
        ]
    ]
    .rename(
        columns={
            "invoice_id": "Factura",
            "model_name": "Nombre",
            "model_version": "Versión",
            "run_id": "Run",
            "manual_review_probability": "Probabilidad",
            "recommendation": "Policy",
            "correlation_id": "Correlation ID",
        }
    )
    .style.format({"Probabilidad": "{:.3f}"})
)

## 5. Fallback: control de riesgo, no juicio sobre el modelo

**Qué estás mirando:** la respuesta segura cuando el modelo no está disponible.

**Por qué importa:** fallback no significa que el modelo sea malo. Significa que el servicio o su dependencia no está disponible y el sistema elige controlar el riesgo antes que inventar evidencia.

**Qué ejecuta esta celda:** simula la respuesta de fallback sin detener el servicio Compose y persiste una recomendación en SQLite y `STATE_PATH`. **⚠ MODIFICA ESTADO** solo en esa auditoría; es idempotente. Recuperación: relee la auditoría por `correlation_id`; si falta, revisa el detalle técnico antes de reintentar.

**Qué deberías comprobar:** `MANUAL_REVIEW`, `source=fallback`, `reason=model_unavailable` y probabilidad `null`.

**Qué significa si falla:** la simulación o la persistencia de evidencia falló; no reemplaces `null` por `1.0`, porque nunca se obtuvo un score.

In [ ]:
# Side-effect contract: resource=isolated SQLite fallback audit and local state; condition=explicit fallback simulation; idempotency=durable action identity; recovery=inspect correlation_id before retrying.
# ⚠ MODIFICA ESTADO: registra fallback sin fabricar una probabilidad.
fallback = fallback_recommendation()


def persist_fallback():
    correlation_id = "notebook-05:INV-10030:fallback"
    insert_model_evaluation(
        DEMO_DB,
        "INV-10030",
        correlation_id=correlation_id,
        recommendation=fallback,
        model_name=None,
        model_version=None,
        run_id=None,
        manual_review_probability=None,
    )
    return {
        "Factura": "INV-10030",
        "Policy": fallback.decision.value,
        "source": fallback.source,
        "reason": fallback.reason,
        "Probabilidad": None,
        "Correlation ID": correlation_id,
    }


fallback_result = run_mutable_action_once(
    f"persist-fallback-INV-10030:{AUDIT_ACTION_SCOPE}", state["completed_actions"], persist_fallback
)
state["evaluations"]["fallback"] = fallback_result
save_state()
display(pd.DataFrame([fallback_result]))

## Idea de cierre

Una versión `champion` define una referencia en Registry. `/health` demuestra qué versión ya usa el runtime Compose; un cambio de alias no lo recarga automáticamente. `/predict` aporta una probabilidad; `ml-policy-v1` entrega una recomendación; la auditoría deja contexto para reconstruir la evaluación. La decisión final sigue siendo una responsabilidad separada.